# NLP Lab 5: Encoder-Only Transformer for Sequence Classification

In this lab, we build an **Encoder-Only Transformer** (similar to the BERT architecture) from scratch using PyTorch's `nn.TransformerEncoder`. We will train it on a procedurally generated sentiment classification corpus to demonstrate how self-attention processes sequences for text classification.

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import math
import random
import itertools

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42)
random.seed(42)
print(f"Using device: {device}")

Using device: cuda


## 1. Generating a Large Toy Corpus

We procedurally generate a dataset of 1,200 synthetic movie reviews (600 positive, 600 negative) by combining different sentiment-bearing adjectives, nouns, and concluding phrases. We then build a vocabulary and pad the sequences for batching.

In [3]:
# Procedural Corpus Generation
pos_adjs = ["amazing", "brilliant", "fantastic", "great", "wonderful", "superb"]
pos_nouns = ["movie", "film", "story", "acting", "cinema", "masterpiece"]
pos_endings = ["loved it", "highly recommend", "a joy to watch", "perfect", "enjoyed every minute"]

neg_adjs = ["terrible", "awful", "boring", "bad", "horrible", "dull"]
neg_nouns = ["movie", "film", "script", "direction", "mess", "disaster"]
neg_endings = ["hated it", "waste of time", "do not watch", "terrible", "disappointing"]

# Generate combinations
pos_sentences = [f"{a} {n} {e}" for a, n, e in itertools.product(pos_adjs, pos_nouns, pos_endings)]
neg_sentences = [f"{a} {n} {e}" for a, n, e in itertools.product(neg_adjs, neg_nouns, neg_endings)]

# Sample 600 of each from the generated pool.
# Each pool contains 180 unique combinations, so sampling must allow repetition.
pos_sentences = random.choices(pos_sentences, k=600)
neg_sentences = random.choices(neg_sentences, k=600)

sentences = pos_sentences + neg_sentences
labels = [1] * 600 + [0] * 600  # 1: Positive, 0: Negative

# Shuffle dataset
combined = list(zip(sentences, labels))
random.shuffle(combined)
sentences, labels = zip(*combined)

# Build Vocabulary
vocab = {"<PAD>": 0, "<UNK>": 1}
for sent in sentences:
    for word in sent.split():
        if word not in vocab:
            vocab[word] = len(vocab)

print(f"Total Sentences: {len(sentences)} | Vocabulary Size: {len(vocab)}")

# Tokenize and Pad sequences
MAX_LEN = 8
def encode(sent):
    tokens = [vocab.get(w, vocab["<UNK>"]) for w in sent.split()]
    return tokens[:MAX_LEN] + [vocab["<PAD>"]] * max(0, MAX_LEN - len(tokens))

X = torch.tensor([encode(s) for s in sentences], dtype=torch.long)
y = torch.tensor(labels, dtype=torch.float32)

# Create DataLoader
dataset = TensorDataset(X, y)
train_loader = DataLoader(dataset, batch_size=32, shuffle=True)

Total Sentences: 1200 | Vocabulary Size: 43


## 2. Transformer Encoder Classification Model

We use `nn.TransformerEncoderLayer` and `nn.TransformerEncoder` to build the network. 

To perform classification, we must aggregate the sequence of hidden states outputted by the encoder into a single vector. We achieve this using **Mean Pooling**, which averages the representations across the sequence dimension `(Batch, Seq_Len, Features) -> (Batch, Features)` before passing it to the final linear layer.

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)).unsqueeze(0)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe

class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size, d_model=128, nhead=4, num_layers=2, num_classes=1, max_len=32):
        super().__init__()
        self.d_model = d_model
        
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_encoder = PositionalEncoding(d_model, max_len=max_len)
        
        # Encoder Layer (batch_first=True aligns with [Batch, Seq_Len, Features])
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, 
            nhead=nhead, 
            dim_feedforward=256, 
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # Final classification head
        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, x, padding_mask=None):
        # 1. Embed and apply Positional Encoding
        x = self.embedding(x) * math.sqrt(self.d_model)
        x = self.pos_encoder(x)
        
        # 2. Pass through Transformer Encoder
        encoded = self.transformer_encoder(x, src_key_padding_mask=padding_mask)
        
        # 3. Mean Pooling: Average the token representations across the sequence length
        pooled = encoded.mean(dim=1) 
        
        # 4. Compute Logits
        logits = self.classifier(pooled).squeeze(-1)
        return logits

## 3. Training the Transformer

We initialize the model and train it using Binary Cross Entropy (`BCEWithLogitsLoss`). To ensure the Transformer's attention mechanism does not extract information from `<PAD>` tokens, we dynamically generate a boolean `padding_mask` for each batch.

In [16]:
# Instantiate the model
model = TransformerClassifier(
    vocab_size=len(vocab),
    d_model=64,
    nhead=2,
    num_layers=2,
    max_len=MAX_LEN
).to(device)

optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.BCEWithLogitsLoss()

print(model)
print("Starting Training...\n")

model.train()
for epoch in range(5):
    total_loss = 0
    correct = 0
    
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        
        # Create padding mask: True where token is <PAD> (index 0)
        padding_mask = (inputs == 0).to(device)
        
        optimizer.zero_grad()
        
        # Forward pass
        logits = model(inputs, padding_mask=padding_mask)
        
        # Compute loss and backward pass
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        # Compute accuracy
        preds = (torch.sigmoid(logits) >= 0.5).float()
        correct += (preds == targets).sum().item()
        
    avg_loss = total_loss / len(train_loader)
    acc = (correct / len(sentences)) * 100
    print(f"Epoch {epoch+1}/5 | Loss: {avg_loss:.4f} | Accuracy: {acc:.2f}%")

TransformerClassifier(
  (embedding): Embedding(43, 64, padding_idx=0)
  (pos_encoder): PositionalEncoding()
  (transformer_encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
        )
        (linear1): Linear(in_features=64, out_features=256, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=256, out_features=64, bias=True)
        (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
        (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (classifier): Linear(in_features=64, out_features=1, bias=True)
)
Starting Training...

Epoch 1/5 | Loss: 0.1434 | Accuracy: 95.92%
Epoch 2/5 | Loss: 0.0052

## 4. Inference Test

Let's test the trained model on new, unseen examples to verify that it learned the sentiment representations.

In [17]:
def predict(text, model, vocab, max_len=MAX_LEN):
    model.eval()
    tokens = [vocab.get(w, vocab["<UNK>"]) for w in text.split()]
    padded = tokens[:max_len] + [vocab["<PAD>"]] * max(0, max_len - len(tokens))
    
    input_tensor = torch.tensor([padded], dtype=torch.long).to(device)
    pad_mask = (input_tensor == 0).to(device)
    
    with torch.no_grad():
        logit = model(input_tensor, padding_mask=pad_mask)
        prob = torch.sigmoid(logit).item()
        
    sentiment = "Positive" if prob >= 0.5 else "Negative"
    print(f"Text: '{text}'")
    print(f"Prediction: {sentiment} ({prob:.4f})\n")

# Run inference
test_sentences = [
    "fantastic cinema highly recommend",
    "terrible script waste of time",
    "amazing acting loved it",
    "boring film do not watch"
]

print("--- Inference Results ---")
for sent in test_sentences:
    predict(sent, model, vocab)

--- Inference Results ---
Text: 'fantastic cinema highly recommend'
Prediction: Positive (0.9669)

Text: 'terrible script waste of time'
Prediction: Negative (0.0141)

Text: 'amazing acting loved it'
Prediction: Positive (0.9659)

Text: 'boring film do not watch'
Prediction: Negative (0.0148)



/home/pc/Desktop/CSE_4122_2K21/nlpLab/lib/python3.12/site-packages/torch/nn/modules/transformer.py:529: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/NestedTensorImpl.cpp:177.)
  output = torch._nested_tensor_from_mask(
